[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/68_noise_schedule_solution.ipynb)

# 🟢 Solution: Diffusion Noise Schedules

Reference solution for three noise schedules used in diffusion models.

Each schedule returns $\bar{\alpha}$, the cumulative product of $(1 - \beta)$ across timesteps. $\bar{\alpha}_t$ controls the signal-to-noise ratio: close to 1 means mostly signal, close to 0 means mostly noise.

**linear**: $\beta_t = \beta_{\text{start}} + (\beta_{\text{end}} - \beta_{\text{start}}) \cdot \frac{t}{T-1}$, then $\bar{\alpha} = \text{cumprod}(1 - \beta)$

**cosine**: $\bar{\alpha}_t = \cos^2\!\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right) \Big/ \cos^2\!\left(\frac{s}{1+s} \cdot \frac{\pi}{2}\right)$, clipped to $[0.0001, 0.9999]$

**sigmoid**: $\beta_t = \text{sigmoid}(t/T \cdot 12 - 6)$, scaled to $[\beta_{\text{start}}, \beta_{\text{end}}]$, then $\bar{\alpha} = \text{cumprod}(1 - \beta)$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch, math

In [ ]:
# ✅ SOLUTION

def noise_schedule(num_timesteps, schedule_type='cosine'):
    T = num_timesteps
    t = torch.arange(T, dtype=torch.float32)
    if schedule_type == 'linear':
        beta_start, beta_end = 1e-4, 0.02
        betas = beta_start + (beta_end - beta_start) * t / (T - 1)
        alpha_bar = torch.cumprod(1 - betas, dim=0)
    elif schedule_type == 'cosine':
        s = 0.008
        f = torch.cos(((t / T + s) / (1 + s)) * (math.pi / 2)) ** 2
        f0 = math.cos((s / (1 + s)) * (math.pi / 2)) ** 2
        alpha_bar = (f / f0).clamp(0.0001, 0.9999)
    elif schedule_type == 'sigmoid':
        beta_start, beta_end = 1e-4, 0.02
        x = t / T * 12 - 6
        sig = 1 / (1 + torch.exp(-x))
        betas = beta_start + (beta_end - beta_start) * (sig - sig.min()) / (sig.max() - sig.min())
        alpha_bar = torch.cumprod(1 - betas, dim=0)
    else:
        raise ValueError(f'Unknown schedule_type: {schedule_type}')
    return alpha_bar

In [ ]:
T = 1000
schedules = ['linear', 'cosine', 'sigmoid']
checkpoints = [0, T // 4, T // 2, 3 * T // 4, T - 1]
labels = ['t=0', 'T/4', 'T/2', '3T/4', 'T-1']

# Header
col_w = 10
header = f"{'t':>6}" + "".join(f"{s:>{col_w}}" for s in schedules)
print(header)
print("-" * len(header))

results = {s: noise_schedule(T, s) for s in schedules}
for label, idx in zip(labels, checkpoints):
    row = f"{label:>6}" + "".join(f"{results[s][idx].item():>{col_w}.4f}" for s in schedules)
    print(row)

In [ ]:
from torch_judge import check
check("noise_schedule")